# Register Tiling GEMM v3 — Bank Conflict Fix (As PAD + Bs XOR Swizzle)

**Runtime → Change runtime type → GPU (T4)**

## 修复内容

### Fix 1: As Padding (2-way → 0-way)

`As[BM][BK]` 中 BK=8，同一 warp 内两个 ty 值间隔 TM=8 行：
- 原版: bank 间距 = 8 × 8 = 64, **64 % 32 = 0** → 同一 bank → 2-way conflict
- 修复: `As[BM][BK+1]`, stride=9, bank 间距 = 8 × 9 = 72, **72 % 32 = 8** → 不同 bank → 无 conflict

为什么 PAD=4 失败: 8 × 12 = 96, 96 % 32 = 0 → 仍然同一 bank。**PAD 不能是 4 的倍数**。

### Fix 2: Bs XOR Swizzle (4-way → ≤2-way)

`Bs[BK][BN]` 行内访问 `Bs[bk][tx*8+tn]`，16 个 tx 间距 8：
- 原版: tx=0,4,8,12 → bank=(tx*8)%32 → 同一 bank → 4-way conflict
- Padding 无法修复行内 conflict（只影响行间距）
- 修复: XOR swizzle `col ^ ((col >> 3) << 1)` 打散 bank 映射
  - 该函数是 [0,127] 上的双射（一一对应），store 和 read 用同一变换
  - 将 4-way → 最大 2-way（16 个地址中仅 3 对 2-way conflict）

### 预期改善

| 指标 | v1 | v3 (预期) |
|------|-----|-----------|
| As read conflict | 2-way | 0-way |
| Bs read conflict | 4-way | ≤2-way |
| 加权平均 | 3.2-way | ~1.2-way |

In [1]:
# 检查 GPU 和 CUDA 版本
!nvidia-smi
!nvcc --version

Mon Apr  6 02:02:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile matmul_reg_tiling_v3.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define BM 128
#define BN 128
#define BK 8
#define TM 8
#define TN 8

// ============================================================
// Fix 1: As padding
// ============================================================
// As[BM][BK] 的 2-way bank conflict 来自:
//   同一 warp 内两个 ty 值间隔 TM=8 行
//   bank 间距 = TM * stride % 32
//   原版: stride=BK=8, 8*8=64, 64%32=0 → 同 bank → 2-way
//   PAD=1: stride=9, 8*9=72, 72%32=8 → 不同 bank → 无 conflict
// 注: PAD=4 无效, 因为 8*12=96, 96%32=0
#define A_PAD 1

// ============================================================
// Fix 2: Bs XOR swizzle
// ============================================================
// Bs[BK][BN] 的 4-way bank conflict 来自 行内 访问模式:
//   regB[tn] = Bs[bk][tx*TN+tn], tx 间距=TN=8
//   tx=0,4,8,12 → (tx*8)%32=0 → 同 bank → 4-way
//   padding 只影响行间距, 无法修复行内 conflict
//
// XOR swizzle: col → col ^ ((col >> 3) << 1)
//   等价于: col → col ^ (tx * 2)  (其中 tx = col / 8)
//   将 {tx=0,4,8,12} 从同一 bank 分散到 4 个不同 bank
//   结果: 16 个地址中最多 3 对 2-way conflict (平均 ~1.4-way)
//   该函数是 [0,127] 上的双射, store 和 read 用同一变换保证正确性
#define BS_SWIZZLE(col) ((col) ^ (((col) >> 3) << 1))

// ============================================================
// v3 kernel: As PAD + Bs XOR swizzle
// ============================================================
__global__ void MatMulRegTilingKernel_v3(const float* __restrict__ A,
                                         const float* __restrict__ B,
                                         float* __restrict__ C,
                                         int M, int N, int K)
{
    const int bx = blockIdx.x;
    const int by = blockIdx.y;
    const int tx = threadIdx.x;
    const int ty = threadIdx.y;

    const int tid = ty * blockDim.x + tx;
    const int numThreads = blockDim.x * blockDim.y;

    // FIX 1: As 加 PAD=1, stride 从 8 变为 9
    __shared__ float As[BM][BK + A_PAD];  // 128 x 9 (原 128 x 8)
    __shared__ float Bs[BK][BN];          // 8 x 128 (布局不变, 通过 swizzle 索引)

    float regC[TM][TN] = {0.0f};
    float regA[TM];
    float regB[TN];

    const int cRow = by * BM;
    const int cCol = bx * BN;

    const float* aPtr = A + cRow * K;
    const float* bPtr = B + cCol;

    const int loadARow = tid / BK;
    const int loadACol = tid % BK;
    const int strideA  = numThreads / BK;

    const int loadBRow = tid / BN;
    const int loadBCol = tid % BN;
    const int strideB  = numThreads / BN;

    for (int k = 0; k < K; k += BK) {
        // 加载 As — PAD 只影响布局, 加载代码不变
        for (int i = 0; i < BM; i += strideA) {
            As[loadARow + i][loadACol] = aPtr[(loadARow + i) * K + k + loadACol];
        }
        // FIX 2: 加载 Bs 时使用 swizzle 索引
        for (int i = 0; i < BK; i += strideB) {
            Bs[loadBRow + i][BS_SWIZZLE(loadBCol)] = bPtr[(k + loadBRow + i) * N + loadBCol];
        }

        __syncthreads();

        for (int bk = 0; bk < BK; ++bk) {
            // As read: PAD 后 stride=9, 两个 ty 间 bank 间距=8, 无 conflict
            for (int tm = 0; tm < TM; ++tm) {
                regA[tm] = As[ty * TM + tm][bk];
            }
            // FIX 2: Bs read 使用相同的 swizzle
            for (int tn = 0; tn < TN; ++tn) {
                regB[tn] = Bs[bk][BS_SWIZZLE(tx * TN + tn)];
            }
            for (int tm = 0; tm < TM; ++tm) {
                for (int tn = 0; tn < TN; ++tn) {
                    regC[tm][tn] += regA[tm] * regB[tn];
                }
            }
        }

        __syncthreads();
    }

    // 写回 C (未优化 global store coalescing, 留作下一步)
    for (int tm = 0; tm < TM; ++tm) {
        int globalRow = cRow + ty * TM + tm;
        for (int tn = 0; tn < TN; ++tn) {
            int globalCol = cCol + tx * TN + tn;
            C[globalRow * N + globalCol] = regC[tm][tn];
        }
    }
}

// ============================================================
// v1 kernel: 原版 (用于对比)
// ============================================================
__global__ void MatMulRegTilingKernel_v1(const float* __restrict__ A,
                                         const float* __restrict__ B,
                                         float* __restrict__ C,
                                         int M, int N, int K)
{
    const int bx = blockIdx.x;
    const int by = blockIdx.y;
    const int tx = threadIdx.x;
    const int ty = threadIdx.y;
    const int tid = ty * blockDim.x + tx;
    const int numThreads = blockDim.x * blockDim.y;

    __shared__ float As[BM][BK];
    __shared__ float Bs[BK][BN];

    float regC[TM][TN] = {0.0f};
    float regA[TM];
    float regB[TN];

    const int cRow = by * BM;
    const int cCol = bx * BN;
    const float* aPtr = A + cRow * K;
    const float* bPtr = B + cCol;

    const int loadARow = tid / BK;
    const int loadACol = tid % BK;
    const int strideA  = numThreads / BK;
    const int loadBRow = tid / BN;
    const int loadBCol = tid % BN;
    const int strideB  = numThreads / BN;

    for (int k = 0; k < K; k += BK) {
        for (int i = 0; i < BM; i += strideA)
            As[loadARow + i][loadACol] = aPtr[(loadARow + i) * K + k + loadACol];
        for (int i = 0; i < BK; i += strideB)
            Bs[loadBRow + i][loadBCol] = bPtr[(k + loadBRow + i) * N + loadBCol];

        __syncthreads();

        for (int bk = 0; bk < BK; ++bk) {
            for (int tm = 0; tm < TM; ++tm)
                regA[tm] = As[ty * TM + tm][bk];
            for (int tn = 0; tn < TN; ++tn)
                regB[tn] = Bs[bk][tx * TN + tn];
            for (int tm = 0; tm < TM; ++tm)
                for (int tn = 0; tn < TN; ++tn)
                    regC[tm][tn] += regA[tm] * regB[tn];
        }
        __syncthreads();
    }

    for (int tm = 0; tm < TM; ++tm)
        for (int tn = 0; tn < TN; ++tn) {
            int globalRow = cRow + ty * TM + tm;
            int globalCol = cCol + tx * TN + tn;
            C[globalRow * N + globalCol] = regC[tm][tn];
        }
}

int main()
{
    int M = 2048, N = 2048, K = 2048;

    size_t bytesA = M * K * sizeof(float);
    size_t bytesB = K * N * sizeof(float);
    size_t bytesC = M * N * sizeof(float);

    float* A     = (float*)malloc(bytesA);
    float* B     = (float*)malloc(bytesB);
    float* C_v1  = (float*)malloc(bytesC);
    float* C_v3  = (float*)malloc(bytesC);

    for (int i = 0; i < M * K; ++i) A[i] = (float)(i % 10) * 0.1f;
    for (int i = 0; i < K * N; ++i) B[i] = (float)(i % 7)  * 0.1f;

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytesA);
    cudaMalloc(&d_B, bytesB);
    cudaMalloc(&d_C, bytesC);
    cudaMemcpy(d_A, A, bytesA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, bytesB, cudaMemcpyHostToDevice);

    dim3 dimBlock(BN / TN, BM / TM);  // (16, 16)
    dim3 dimGrid(N / BN, M / BM);     // (16, 16)

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float ms = 0;

    // ========== v1: 原版 ==========
    MatMulRegTilingKernel_v1<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, M, N, K);
    cudaDeviceSynchronize();
    cudaEventRecord(start);
    MatMulRegTilingKernel_v1<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, M, N, K);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    float v1_ms = ms;
    cudaMemcpy(C_v1, d_C, bytesC, cudaMemcpyDeviceToHost);

    // ========== v3: bank conflict fix ==========
    MatMulRegTilingKernel_v3<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, M, N, K);
    cudaDeviceSynchronize();
    cudaEventRecord(start);
    MatMulRegTilingKernel_v3<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, M, N, K);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    float v3_ms = ms;
    cudaMemcpy(C_v3, d_C, bytesC, cudaMemcpyDeviceToHost);

    // ========== 验证 ==========
    float max_err = 0.0f;
    for (int i = 0; i < M * N; ++i) {
        float err = fabsf(C_v1[i] - C_v3[i]);
        if (err > max_err) max_err = err;
    }

    double gflops_v1 = 2.0 * M * N * K / (v1_ms * 1e-3) / 1e9;
    double gflops_v3 = 2.0 * M * N * K / (v3_ms * 1e-3) / 1e9;

    printf("=== Register Tiling GEMM: v1 vs v3 (bank conflict fix) ===\n");
    printf("Matrix: %dx%dx%d\n", M, N, K);
    printf("Block tile: %dx%d, Thread tile: %dx%d, K tile: %d\n", BM, BN, TM, TN, BK);
    printf("\n");
    printf("v1 (original):          %.3f ms  (%.1f GFLOPS)\n", v1_ms, gflops_v1);
    printf("v3 (PAD+swizzle fix):   %.3f ms  (%.1f GFLOPS)\n", v3_ms, gflops_v3);
    printf("Speedup: %.2fx\n", v1_ms / v3_ms);
    printf("Max diff (v1 vs v3): %e\n", max_err);
    if (max_err < 1e-3f)
        printf("PASSED\n");
    else
        printf("FAILED\n");

    cudaEventDestroy(start); cudaEventDestroy(stop);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(A); free(B); free(C_v1); free(C_v3);
    return 0;
}

Writing matmul_reg_tiling_v3.cu


In [3]:
!nvcc -O3 -lineinfo -arch=sm_75 matmul_reg_tiling_v3.cu -o matmul_reg_tiling_v3
print("编译成功!")

编译成功!


In [4]:
# 运行对比测试
!./matmul_reg_tiling_v3

=== Register Tiling GEMM: v1 vs v3 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):          8.378 ms  (2050.5 GFLOPS)
v3 (PAD+swizzle fix):   5.713 ms  (3007.2 GFLOPS)
Speedup: 1.47x
Max diff (v1 vs v3): 0.000000e+00
PASSED


---
## Part 1: Nsight Systems — 系统级 profile

In [5]:
# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')
!chmod -R +x /content/drive/MyDrive/cuda/nsys_tool

NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} --version

Mounted at /content/drive
NVIDIA Nsight Systems version 2024.5.1.113-245134619542v0


In [6]:
# nsys profile
NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} profile \
     --stats=true \
     --force-overwrite=true \
     -o nsys_reg_tiling_v3 \
     ./matmul_reg_tiling_v3

=== Register Tiling GEMM: v1 vs v3 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):          9.010 ms  (1906.8 GFLOPS)
v3 (PAD+swizzle fix):   6.154 ms  (2791.5 GFLOPS)
Speedup: 1.46x
Max diff (v1 vs v3): 0.000000e+00
PASSED
Generating '/tmp/nsys-report-ecb0.qdstrm'
[1/8] [========================100%] nsys_reg_tiling_v3.nsys-rep
[2/8] [========================100%] nsys_reg_tiling_v3.sqlite
[3/8] Executing 'nvtx_sum' stats report
SKIPPED: /content/nsys_reg_tiling_v3.sqlite does not contain NV Tools Extension (NVTX) data.
[4/8] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)     Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  -----------  --------  -----------  ------------  ----------------------
     75.0      546,474,959         14  39,033,925.6  9,250,972.0   305,927  343,861,224  90,147,346.0  poll         

In [7]:
# Kernel 执行统计
NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} stats --force-export=true --report cuda_gpu_kern_sum nsys_reg_tiling_v3.nsys-rep

Generating SQLite file nsys_reg_tiling_v3.sqlite from nsys_reg_tiling_v3.nsys-rep
Processing [nsys_reg_tiling_v3.sqlite] with [/content/drive/MyDrive/cuda/nsys_tool/host-linux-x64/reports/cuda_gpu_kern_sum.py]... 

 ** CUDA GPU Kernel Summary (cuda_gpu_kern_sum):

 Time (%)  Total Time (ns)  Instances   Avg (ns)     Med (ns)    Min (ns)   Max (ns)   StdDev (ns)                                       Name                                     
 --------  ---------------  ---------  -----------  -----------  ---------  ---------  -----------  ------------------------------------------------------------------------------
     59.5       17,990,114          2  8,995,057.0  8,995,057.0  8,992,161  8,997,953      4,095.6  MatMulRegTilingKernel_v1(const float *, const float *, float *, int, int, int)
     40.5       12,262,249          2  6,131,124.5  6,131,124.5  6,118,868  6,143,381     17,333.3  MatMulRegTilingKernel_v3(const float *, const float *, float *, int, int, int)



---
## Part 2: Nsight Compute — v3 kernel 深度分析

重点验证:
1. shared memory bank conflict 是否大幅降低
2. MIO stall 是否降低
3. eligible warps 是否增加
4. compute/memory throughput 是否提升

In [8]:
# Memory Workload — 重点看 bank conflict 是否降低
!ncu --section MemoryWorkloadAnalysis \
     --section MemoryWorkloadAnalysis_Tables \
     --kernel-name MatMulRegTilingKernel_v3 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v3

==PROF== Connected to process 9450 (/content/matmul_reg_tiling_v3)
==PROF== Profiling "MatMulRegTilingKernel_v3": 0%....50%....100% - 27 passes
=== Register Tiling GEMM: v1 vs v3 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):          14.958 ms  (1148.5 GFLOPS)
v3 (PAD+swizzle fix):   5837.959 ms  (2.9 GFLOPS)
Speedup: 0.00x
Max diff (v1 vs v3): 0.000000e+00
PASSED
==PROF== Disconnected from process 9450
[9450] matmul_reg_tiling_v3@127.0.0.1
  MatMulRegTilingKernel_v3(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Memory Workload Analysis
    ----------------- ----------- ------------
    Metric Name       Metric Unit Metric Value
    ----------------- ----------- ------------
    Memory Throughput     Gbyte/s        22.78
    Mem Busy                    %        36.04
    Max Bandwidth               %        63.89
    L1/TEX Hit Rate      

In [9]:
# Occupancy + Launch Stats
!ncu --section Occupancy \
     --section LaunchStats \
     --kernel-name MatMulRegTilingKernel_v3 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v3

==PROF== Connected to process 9557 (/content/matmul_reg_tiling_v3)
==PROF== Profiling "MatMulRegTilingKernel_v3": 0%....50%....100% - 1 pass
=== Register Tiling GEMM: v1 vs v3 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):          14.604 ms  (1176.4 GFLOPS)
v3 (PAD+swizzle fix):   815.689 ms  (21.1 GFLOPS)
Speedup: 0.02x
Max diff (v1 vs v3): 0.000000e+00
PASSED
==PROF== Disconnected from process 9557
[9557] matmul_reg_tiling_v3@127.0.0.1
  MatMulRegTilingKernel_v3(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Launch Statistics
    -------------------------------- --------------- ---------------
    Metric Name                          Metric Unit    Metric Value
    -------------------------------- --------------- ---------------
    Block Size                                                   256
    Function Cache Configuration         

In [10]:
# Scheduler + Warp Stall
!ncu --section SchedulerStats \
     --section WarpStateStats \
     --kernel-name MatMulRegTilingKernel_v3 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v3

==PROF== Connected to process 9623 (/content/matmul_reg_tiling_v3)
==PROF== Profiling "MatMulRegTilingKernel_v3": 0%....50%....100% - 8 passes
=== Register Tiling GEMM: v1 vs v3 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):          8.535 ms  (2012.8 GFLOPS)
v3 (PAD+swizzle fix):   2335.562 ms  (7.4 GFLOPS)
Speedup: 0.00x
Max diff (v1 vs v3): 0.000000e+00
PASSED
==PROF== Disconnected from process 9623
[9623] matmul_reg_tiling_v3@127.0.0.1
  MatMulRegTilingKernel_v3(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Scheduler Statistics
    ---------------------------- ----------- ------------
    Metric Name                  Metric Unit Metric Value
    ---------------------------- ----------- ------------
    One or More Eligible                   %        48.12
    Issued Warp Per Scheduler                        0.48
    No Eligible        

In [11]:
# Speed of Light + Compute
!ncu --section SpeedOfLight \
     --section ComputeWorkloadAnalysis \
     --kernel-name MatMulRegTilingKernel_v3 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v3

==PROF== Connected to process 9692 (/content/matmul_reg_tiling_v3)
==PROF== Profiling "MatMulRegTilingKernel_v3": 0%....50%....100% - 8 passes
=== Register Tiling GEMM: v1 vs v3 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):          14.600 ms  (1176.7 GFLOPS)
v3 (PAD+swizzle fix):   2424.932 ms  (7.1 GFLOPS)
Speedup: 0.01x
Max diff (v1 vs v3): 0.000000e+00
PASSED
==PROF== Disconnected from process 9692
[9692] matmul_reg_tiling_v3@127.0.0.1
  MatMulRegTilingKernel_v3(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       585.00
    Elapsed Cycles                cycl

In [12]:
# Source Counters
!ncu --section SourceCounters \
     --kernel-name MatMulRegTilingKernel_v3 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v3

==PROF== Connected to process 9760 (/content/matmul_reg_tiling_v3)
==PROF== Profiling "MatMulRegTilingKernel_v3": 0%....50%....100% - 5 passes
=== Register Tiling GEMM: v1 vs v3 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):          11.058 ms  (1553.5 GFLOPS)
v3 (PAD+swizzle fix):   2404.170 ms  (7.1 GFLOPS)
Speedup: 0.00x
Max diff (v1 vs v3): 0.000000e+00
PASSED
==PROF== Disconnected from process 9760
[9760] matmul_reg_tiling_v3@127.0.0.1
  MatMulRegTilingKernel_v3(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Source Counters
    ------------------------- ----------- ------------
    Metric Name               Metric Unit Metric Value
    ------------------------- ----------- ------------
    Branch Instructions Ratio           %         0.03
    Branch Instructions              inst   12,062,720
    Branch Efficiency                   % 

In [13]:
# 导出完整 ncu-rep 报告
!ncu --set full \
     --kernel-name MatMulRegTilingKernel_v3 \
     --launch-skip 1 --launch-count 1 \
     -o reg_tiling_v3_profile \
     ./matmul_reg_tiling_v3

print("\n报告已保存为 reg_tiling_v3_profile.ncu-rep")

==PROF== Connected to process 9842 (/content/matmul_reg_tiling_v3)
==PROF== Profiling "MatMulRegTilingKernel_v3": 0%....50%....100% - 31 passes
=== Register Tiling GEMM: v1 vs v3 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):          13.705 ms  (1253.5 GFLOPS)
v3 (PAD+swizzle fix):   8208.908 ms  (2.1 GFLOPS)
Speedup: 0.00x
Max diff (v1 vs v3): 0.000000e+00
PASSED
==PROF== Disconnected from process 9842
==PROF== Report: /content/reg_tiling_v3_profile.ncu-rep

报告已保存为 reg_tiling_v3_profile.ncu-rep


In [14]:
# 下载报告文件
from google.colab import files
files.download('reg_tiling_v3_profile.ncu-rep')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 对比检查清单

| 指标 | v1 (原版) | v2 (PAD=4, 失败) | v3 (PAD+swizzle) | 改善? |
|------|-----------|-------------------|-------------------|-------|
| Kernel 时间 (ms) | 8.350 | 8.465 | | |
| GFLOPS | 2057.6 | 2029.5 | | |
| Bank conflict (shared load) | 3.2-way, 50% | 3.2-way, 50% | | |
| Bank conflict (shared store) | 无 | 1.5-way | | |
| MIO stall cycles | 7.0 / 12.2 (57.6%) | 7.2 / 12.4 (58.6%) | | |
| Eligible warps / scheduler | 0.42 | 0.41 | | |
| No Eligible % | 68.71% | 69.14% | | |
| Compute (SM) Throughput | 40.68% | ~40% | | |
| Shared Mem Per Block (KB) | 8.19 | 10.24 | | |
| Occupancy (theoretical) | 50% | 50% | | |

### XOR Swizzle 数学验证

Bs read `Bs[bk][BS_SWIZZLE(tx*8+tn)]`，对 tn=0，16 个 tx 的 bank 映射:

```
原版:        tx: 0→0,  1→8,  2→16, 3→24, 4→0,  5→8,  6→16, 7→24, ...  (4-way)
XOR swizzle: tx: 0→0,  1→10, 2→20, 3→30, 4→4,  5→2,  6→12, 7→18,
                 8→16, 9→26, 10→4, 11→6, 12→24, 13→18, 14→12, 15→2
冲突: {5,15}→2, {6,14}→12, {7,13}→18  (最多 2-way)
```